# Distogram Distribution: Real vs Imputed Waters

Compare the **spatial environment** of real (≥ 3 H-bond) vs geometrically imputed waters,
measured relative to polymer residue centers (Cα for protein, sugar C4′ / base center for nucleics).

**Statistics computed per water:**
| Statistic | Description |
|---|---|
| kNN distance | Distance to the k-th nearest residue center (k = 1, 2, 3, 5, 10) |
| N(r) | Number of residue centers within radius r (r = 3–12 Å) |
| Centroid distance | Distance to the center of mass of all residue centers |
| Water–water distance | Distance to the nearest other water of the same kind |

All helpers live in [`distogram_helpers.py`](distogram_helpers.py).
All plots are styled for ICML submission (9 pt serif, no top/right spines, 300 dpi).

---
## Part 0 — Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Add both the current directory and its parent (imputation/) to sys.path
_here = Path('.').resolve()
sys.path.insert(0, str(_here))
sys.path.insert(0, str(_here.parent))

from distogram_helpers import (
    # Structure accessors
    get_residue_coords, SOLVENT_MOL_TYPES, POLYMER_MOL_TYPES,
    # Distance computations
    knn_distances, radius_neighbor_counts, centroid_distances,
    self_nearest_neighbor_distances,
    # Structure loading
    load_gt_and_imputed, analyze_pdb, collect_results,
    # Aggregation
    concat_field,
    # Plotting
    set_paper_style, ecdf_xy,
    plot_knn_violins, plot_radius_count_ecdfs, plot_ecdf_comparison,
)

set_paper_style()  # apply once — affects all figures in this notebook

In [ ]:
# ── PDB ID list (swap this out for a different validation set) ──────────────
# To use a different set: replace the path or replace pdb_ids directly.
PDB_LIST_FILE = Path('../easy.txt')  # <-- change this to switch val sets

pdb_ids = PDB_LIST_FILE.read_text().split()
print(f'{len(pdb_ids)} PDB IDs:', pdb_ids[:6], '...')

In [ ]:
# ── Analysis hyperparameters ─────────────────────────────────────────────────
K_LIST      = [1, 2, 3, 5, 10]     # kNN neighbors to query
RADIUS_LIST = [3, 4, 5, 6, 8, 10, 12]  # Å radii for N(r)

MAX_HBOND_LENGTH   = 3.5   # Å — used for imputation
SOLVENT_CLASH_RAD  = 3.0   # Å — solvent-solvent clash filter
MIN_HBONDS_GT      = 3     # minimum H-bonds for a real water to be kept

# Per-element minimum water-O distance (Å) for protein-clash filtering.
# None = use Bondi 1964 defaults (C: 3.10, S/P: 3.20, N/O/F: exempt).
# Override specific elements, e.g.: {"C": 2.8, "S": 3.0}
ATOM_CLASH_DISTS   = None


---
## Part 1 — Representative atoms: what `atom_center` stores

Each `Structure` residue carries `atom_center`, an **absolute atom index** into `structure.atoms`.
- For protein residues → Cα
- For HOH / imputed waters → O (the only atom)

We use this field throughout instead of searching atom names, so the accessor is O(residues).

In [ ]:
from boltzgen.data.data import Structure
from boltzgen.data import const
from basic_helpers import resolve_npz_path
from pathlib import Path

NPZ_ROOT = Path('/data/rbg/shared/datasets/processed_rcsb/rcsb_solvents/structures')
DEMO_PDB = pdb_ids[0]

npz_path  = resolve_npz_path(DEMO_PDB, NPZ_ROOT)
gt_raw    = Structure.load(npz_path)
gt_raw    = gt_raw.to_one_solvent_per_chain(gt_raw)

print('Residue dtype fields:', gt_raw.residues.dtype.names)

In [ ]:
# Show atom_center for the first few residues of each mol_type
mol_type_names = {v: k for k, v in const.chain_type_ids.items()}

for chain in gt_raw.chains:
    rs = gt_raw.residues[chain['res_idx']:chain['res_idx'] + min(chain['res_num'], 3)]
    mol = mol_type_names[chain['mol_type']]
    for res in rs:
        ci = int(res['atom_center'])
        atom_name = gt_raw.atoms['name'][ci]
        coord     = gt_raw.atoms['coords'][ci]
        print(f"mol={mol:<10} res={res['name']:<5}  atom_center → {atom_name:4}  coords={coord}")
    break  # show one chain; remove break to see all

---
## Part 2 — Single-PDB walkthrough

Step through each computation for one structure so every intermediate is inspectable.

In [ ]:
gt_real, imputed, gt_stripped = load_gt_and_imputed(
    DEMO_PDB,
    max_hbond_length  = MAX_HBOND_LENGTH,
    solvent_clash_rad = SOLVENT_CLASH_RAD,
    min_hbonds_gt     = MIN_HBONDS_GT,
    atom_clash_dists  = ATOM_CLASH_DISTS,
)
print(f'GT real waters : {gt_real.count_solvents()}')
print(f'Imputed waters : {imputed.count_solvents()}')
print(f'Polymer residues: {gt_stripped.count_residues() if hasattr(gt_stripped, "count_residues") else len(gt_stripped.residues)}')


In [ ]:
# Polymer residue centers (Cα etc.) — the reference set for all distance metrics
ref_coords = get_residue_coords(gt_stripped, mol_types=POLYMER_MOL_TYPES)

# Water representative atoms (O)
real_coords    = get_residue_coords(gt_real,   mol_types=SOLVENT_MOL_TYPES)
imputed_coords = get_residue_coords(imputed,   mol_types=SOLVENT_MOL_TYPES)

print(f'Reference residue centers : {ref_coords.shape}')
print(f'Real water coords          : {real_coords.shape}')
print(f'Imputed water coords       : {imputed_coords.shape}')

In [ ]:
# kNN distances: shape (n_waters, len(K_LIST))
real_knn    = knn_distances(real_coords,    ref_coords, K_LIST)
imputed_knn = knn_distances(imputed_coords, ref_coords, K_LIST)

print('real_knn shape   :', real_knn.shape)
print('imputed_knn shape:', imputed_knn.shape)
print('\nMedian nearest-1 distance (Å):')
print(f'  real    = {np.nanmedian(real_knn[:,0]):.2f}')
print(f'  imputed = {np.nanmedian(imputed_knn[:,0]):.2f}')

In [ ]:
fig = plot_knn_violins(real_knn, imputed_knn, K_LIST)
plt.show()

In [ ]:
# Radius counts: N(r) for each water
real_radius    = radius_neighbor_counts(real_coords,    ref_coords, RADIUS_LIST)
imputed_radius = radius_neighbor_counts(imputed_coords, ref_coords, RADIUS_LIST)

print('real_radius shape   :', real_radius.shape)
print('Median N(r=5 Å) real   :', np.median(real_radius[:, RADIUS_LIST.index(5)]))
print('Median N(r=5 Å) imputed:', np.median(imputed_radius[:, RADIUS_LIST.index(5)]))

In [ ]:
fig = plot_radius_count_ecdfs(real_radius, imputed_radius, RADIUS_LIST)
plt.show()

In [ ]:
# Distance to the centroid of all polymer residue centers
real_centroid    = centroid_distances(real_coords,    ref_coords)
imputed_centroid = centroid_distances(imputed_coords, ref_coords)

print(f'Median centroid dist real    : {np.nanmedian(real_centroid):.2f} Å')
print(f'Median centroid dist imputed : {np.nanmedian(imputed_centroid):.2f} Å')

In [ ]:
fig = plot_ecdf_comparison(
    real_centroid, imputed_centroid,
    xlabel='distance to residue-center centroid (Å)',
    title=f'Centroid distance — {DEMO_PDB}',
)
plt.show()

In [ ]:
# Water–water nearest-neighbor distance
real_ww    = self_nearest_neighbor_distances(real_coords)
imputed_ww = self_nearest_neighbor_distances(imputed_coords)

print(f'Median ww distance real    : {np.nanmedian(real_ww):.2f} Å')
print(f'Median ww distance imputed : {np.nanmedian(imputed_ww):.2f} Å')

In [ ]:
fig = plot_ecdf_comparison(
    real_ww, imputed_ww,
    xlabel='distance to nearest same-kind water (Å)',
    title=f'Water–water NN distance — {DEMO_PDB}',
)
plt.show()

---
## Part 3 — Multi-PDB analysis

Run `analyze_pdb` for every entry in `pdb_ids` and collect results.
Takes ~30 s per structure. Results are kept as a list of dicts; use `concat_field`
to pool across all entries.

In [ ]:
results = collect_results(
    pdb_ids,
    K_LIST,
    RADIUS_LIST,
    max_hbond_length  = MAX_HBOND_LENGTH,
    solvent_clash_rad = SOLVENT_CLASH_RAD,
    min_hbonds_gt     = MIN_HBONDS_GT,
    atom_clash_dists  = ATOM_CLASH_DISTS,
)
print(f'\nSuccessfully processed {len(results)}/{len(pdb_ids)} structures.')


In [ ]:
# Pool results across all PDB IDs
all_real_knn       = concat_field(results, 'real_knn')
all_imputed_knn    = concat_field(results, 'imputed_knn')
all_real_radius    = concat_field(results, 'real_radius')
all_imputed_radius = concat_field(results, 'imputed_radius')
all_real_centroid    = concat_field(results, 'real_centroid')
all_imputed_centroid = concat_field(results, 'imputed_centroid')
all_real_ww    = concat_field(results, 'real_ww')
all_imputed_ww = concat_field(results, 'imputed_ww')

n_real_total    = int(sum(r['n_real']    for r in results))
n_imputed_total = int(sum(r['n_imputed'] for r in results))
print(f'Total real waters    : {n_real_total}')
print(f'Total imputed waters : {n_imputed_total}')

---
## Part 4 — kNN distance curves

For each k ∈ {1, 2, 3, 5, 10}, violin shows the distribution of distances to the
k-th nearest polymer residue center, pooled across all waters and all structures.

**Expected**: imputed waters should sit closer to residues than real waters, since
they are placed geometrically from H-bond donor/acceptor triples.

In [ ]:
fig = plot_knn_violins(all_real_knn, all_imputed_knn, K_LIST)
plt.show()

---
## Part 5 — Radius counts N(r)

N(r) = number of polymer residue centers within r Å of each water.
ECDFs plotted for r = 3, 4, 5, 6, 8, 10, 12 Å.

**Expected**: imputed waters should have higher N(r) at small r (they are placed near
residues by construction); real waters may show a wider spread.

In [ ]:
fig = plot_radius_count_ecdfs(all_real_radius, all_imputed_radius, RADIUS_LIST)
plt.show()

---
## Part 6 — Distance to residue-center centroid

Tests whether imputed waters are more surface-like (large centroid distance) or
more buried (small centroid distance) than real waters.

In [ ]:
fig = plot_ecdf_comparison(
    all_real_centroid,
    all_imputed_centroid,
    xlabel='distance to residue-center centroid (Å)',
    title='Distance to protein centroid: real vs imputed',
)
plt.show()

---
## Part 7 — Water–water nearest-neighbor distances

For each water, the distance to its nearest other water of the same kind
(real–real or imputed–imputed).

**Expected**: imputed waters may cluster more tightly near H-bond donor/acceptor
sites; real waters are more spread out.

In [ ]:
fig = plot_ecdf_comparison(
    all_real_ww,
    all_imputed_ww,
    xlabel='distance to nearest same-kind water (Å)',
    title='Water–water nearest-neighbor distance: real vs imputed',
)
plt.show()

---
## Part 8 — Repeat analysis with per-element VdW protein-clash filter

The `filter_solvent_clashes` function now uses per-element VdW radii to check
protein–water clashes instead of a single `protein_clash_rad` cutoff. Atom-specific
thresholds (C: 3.10 Å, S/P: 3.20 Å; N/O/F: exempt as H-bond partners) replace
both the old `protein_clash_rad` and `center_clash_rad` parameters. This filtering
is now always applied — Parts 3 and 8 therefore use the same filter, so this section
confirms the results are stable and serves as a reference copy with the final settings.


In [ ]:
results_filtered = collect_results(
    pdb_ids,
    K_LIST,
    RADIUS_LIST,
    max_hbond_length  = MAX_HBOND_LENGTH,
    solvent_clash_rad = SOLVENT_CLASH_RAD,
    min_hbonds_gt     = MIN_HBONDS_GT,
    atom_clash_dists  = ATOM_CLASH_DISTS,
)
print(f'\nProcessed {len(results_filtered)}/{len(pdb_ids)} structures.')


In [ ]:
f_real_knn       = concat_field(results_filtered, 'real_knn')
f_imputed_knn    = concat_field(results_filtered, 'imputed_knn')
f_real_radius    = concat_field(results_filtered, 'real_radius')
f_imputed_radius = concat_field(results_filtered, 'imputed_radius')
f_real_centroid    = concat_field(results_filtered, 'real_centroid')
f_imputed_centroid = concat_field(results_filtered, 'imputed_centroid')
f_real_ww    = concat_field(results_filtered, 'real_ww')
f_imputed_ww = concat_field(results_filtered, 'imputed_ww')

n_real_f    = int(sum(r['n_real']    for r in results_filtered))
n_imputed_f = int(sum(r['n_imputed'] for r in results_filtered))
print(f'Total real waters    : {n_real_f}')
print(f'Total imputed waters : {n_imputed_f}  (was {n_imputed_total} before center filter)')

### kNN distance curves (center-filtered)

In [ ]:
fig = plot_knn_violins(f_real_knn, f_imputed_knn, K_LIST)
plt.show()

### N(r) ECDFs (center-filtered)

In [ ]:
fig = plot_radius_count_ecdfs(f_real_radius, f_imputed_radius, RADIUS_LIST)
plt.show()

### Centroid distances (center-filtered)

In [ ]:
fig = plot_ecdf_comparison(
    f_real_centroid, f_imputed_centroid,
    xlabel='distance to residue-center centroid (Å)',
    title='Distance to protein centroid: real vs imputed (center-filtered)',
)
plt.show()

### Water–water nearest-neighbor distances (center-filtered)

In [ ]:
fig = plot_ecdf_comparison(
    f_real_ww, f_imputed_ww,
    xlabel='distance to nearest same-kind water (Å)',
    title='Water–water NN distance: real vs imputed (center-filtered)',
)
plt.show()